# Getting started with libstb

Install the requirements first: `pip install -r requirements.txt`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import libstb

libstb.__file__

## Make an image

`libstb.Image(array)` wraps a writable, C-contiguous `uint8` array without copying it.

In [ ]:
y, x = np.mgrid[0:200, 0:320]
r = (x * 255 // 319).astype(np.uint8)
g = (y * 255 // 199).astype(np.uint8)
b = np.full_like(r, 128)
img = libstb.Image(np.dstack([r, g, b]))
img.width, img.height, img.channels

In [ ]:
plt.imshow(img.numpy())
plt.axis('off');

## Encode in memory

`to_png`, `to_jpg`, `to_bmp` and `to_tga` return the file's bytes. Each has its own settings.

In [ ]:
sizes = {
    "png (compression=9)": len(img.to_png(compression=9)),
    "jpg (quality=20)": len(img.to_jpg(quality=20)),
    "jpg (quality=90)": len(img.to_jpg(quality=90)),
    "bmp": len(img.to_bmp()),
}
for name, n in sizes.items():
    print(f"{name:22s} {n:>8,} bytes")

## Decode again

`Image.open` takes a path or bytes. `ImageInfo.read` reads only the header.

In [ ]:
data = img.to_png()
print(libstb.ImageInfo.read(data))
back = libstb.Image.open(data)
print("lossless round trip:", np.array_equal(back.numpy(), img.numpy()))

jpg = libstb.Image.open(img.to_jpg(quality=20))
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(back.numpy()); axes[0].set_title("PNG (exact)")
axes[1].imshow(jpg.numpy()); axes[1].set_title("JPEG quality 20")
for a in axes: a.axis("off")

## Channels and flipping

Ask for a channel count or a vertical flip while decoding.

In [ ]:
rgba = libstb.Image.open(data, channels=4)
flipped = libstb.Image.open(data, flip=True)
print("rgba channels:", rgba.channels, "| alpha:", rgba.numpy()[0, 0, 3])
plt.imshow(flipped.numpy()); plt.axis("off");